# Feature Engineering — Environmental Anomaly Detector

Extract LST, spectral reflectance, rolling statistics, and SPC control limits
for anomaly detection models.

In [ ]:
import pandas as pd
import numpy as np
import rasterio
from scipy.signal import savgol_filter

## Feature Engineering Steps

1. Extract Land Surface Temperature (LST) from thermal bands
2. Compute spectral reflectance indices from satellite imagery
3. Calculate rolling statistics (mean, std, z-score) over sliding windows
4. Derive SPC control limits (UCL, LCL) using Western Electric rules
5. Assemble final feature matrix

In [ ]:
# Load sensor data and compute rolling features
df = pd.read_csv('../data/raw/sensor_readings.csv', parse_dates=['timestamp'])
df = df.sort_values(['sensor_id', 'timestamp'])

# Rolling statistics per sensor
for window in [6, 12, 24]:
    for col in ['temperature', 'humidity', 'pm25']:
        grp = df.groupby('sensor_id')[col]
        df[f'{col}_roll{window}_mean'] = grp.transform(lambda x: x.rolling(window).mean())
        df[f'{col}_roll{window}_std'] = grp.transform(lambda x: x.rolling(window).std())

# Z-score relative to rolling baseline
for col in ['temperature', 'humidity', 'pm25']:
    df[f'{col}_zscore'] = (df[col] - df[f'{col}_roll24_mean']) / (df[f'{col}_roll24_std'] + 1e-8)

print(f'Feature matrix shape: {df.shape}')

In [ ]:
# SPC control limits (3-sigma)
spc_features = []
for sid, group in df.groupby('sensor_id'):
    for col in ['temperature', 'pm25']:
        mu = group[col].mean()
        sigma = group[col].std()
        group[f'{col}_ucl'] = mu + 3 * sigma
        group[f'{col}_lcl'] = mu - 3 * sigma
        group[f'{col}_breach'] = ((group[col] > mu + 3 * sigma) | (group[col] < mu - 3 * sigma)).astype(int)
    spc_features.append(group)

df_spc = pd.concat(spc_features)
df_spc.to_parquet('../data/processed/feature_matrix.parquet', index=False)
print(f'Saved feature matrix: {df_spc.shape}')